<a href="https://colab.research.google.com/github/Travis-Bickle10/bangalore-air-quality/blob/main/01_openaq_fetch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
print(response.status_code)
print(response.text[:500])

422
"[{'type': 'less_than_equal', 'loc': ('query', 'radius'), 'msg': 'Input should be less than or equal to 25000', 'input': '50000', 'ctx': {'le': 25000}}]"


In [3]:
from google.colab import userdata
import requests
import pandas as pd

key = userdata.get("OPENAQ_API_KEY")
print("Key loaded:", bool(key))
print("Pandas version:", pd.__version__)
print("Requests version:", requests.__version__)

Key loaded: True
Pandas version: 2.2.2
Requests version: 2.32.4


In [8]:
response = requests.get(
    "https://api.openaq.org/v3/locations",
    headers={"X-API-Key": key},
    params={
        "coordinates": "12.9716,77.5946",
        "radius": 25000,                  # max allowed is 25km
        "limit": 20
    }
)

data = response.json()
print(f"Status: {response.status_code}")
print(f"Total stations found: {data['meta']['found']}")

stations = []
for loc in data["results"]:
    stations.append({
        "id":         loc["id"],
        "name":       loc["name"],
        "country":    loc["country"]["code"],
        "parameters": [s["parameter"]["name"] for s in loc["sensors"]],
        "from":       loc["datetimeFirst"]["local"][:10] if loc["datetimeFirst"] else "N/A",
        "to":         loc["datetimeLast"]["local"][:10] if loc["datetimeLast"] else "N/A",
        "latitude":   loc["coordinates"]["latitude"],
        "longitude":  loc["coordinates"]["longitude"],
    })

df_stations = pd.DataFrame(stations)
df_stations

Status: 200
Total stations found: >20


,id,name,country,parameters,from,to,latitude,longitude
0,412,"Peenya, Bengaluru - KSPCB",IN,"[co, no2, pm25, so2]",2016-03-22,2018-02-22,13.033900,77.513211
1,594,"BTM Layout, Bengaluru - KSPCB",IN,"[co, no2, o3, pm25, so2]",2016-03-22,2018-02-22,12.912811,77.609219
2,797,City Railway Station - KSPCB,IN,"[co, no2, pm10, so2]",2016-03-21,2018-02-22,12.977347,77.570697
3,2589,SaneguravaHalli - KSPCB,IN,"[co, no2, pm10, so2]",2016-03-22,2018-02-22,12.991669,77.545831
4,2592,"BWSSB Kadabesanahalli, Bengaluru - KSPCB",IN,"[co, no2, o3, pm25, so2]",2016-03-22,2018-02-22,12.938906,77.697272
5,5547,"BWSSB Kadabesanahalli, Bengaluru - CPCB",IN,"[co, no2, o3, pm10, pm25, so2]",2018-03-09,2022-08-02,12.935205,77.681449
6,5548,"BTM Layout, Bengaluru - CPCB",IN,"[co, co, no, no2, no2, nox, o3, o3, pm10, pm10...",2018-03-09,2026-03-29,12.913522,77.595080
7,5574,"City Railway Station, Bengaluru - KSPCB",IN,"[co, co, no, no2, no2, nox, pm10, pm10, so2, so2]",2018-03-09,2026-03-28,12.975684,77.566075
8,5607,"Peenya, Bengaluru - CPCB",IN,"[co, co, no, no2, no2, nox, o3, o3, pm10, pm10...",2018-03-09,2026-03-29,13.027020,77.494094
9,5644,"Sanegurava Halli, Bengaluru - KSPCB",IN,"[co, co, no, no2, no2, nox, pm10, pm10, relati...",2018-03-09,2026-03-29,12.990328,77.543138


In [9]:
# Filter to Indian stations only with relevant pollutants
target_pollutants = {"pm25", "pm10", "no2", "so2", "o3", "co"}

df_india = df_stations[df_stations["country"] == "IN"].copy()
df_india["useful_params"] = df_india["parameters"].apply(
    lambda p: [x for x in p if x in target_pollutants]
)
df_india = df_india[df_india["useful_params"].str.len() > 0]

print(f"Indian stations with relevant pollutants: {len(df_india)}")
df_india[["id", "name", "useful_params", "from", "to"]]

Indian stations with relevant pollutants: 20


,id,name,useful_params,from,to
0,412,"Peenya, Bengaluru - KSPCB","[co, no2, pm25, so2]",2016-03-22,2018-02-22
1,594,"BTM Layout, Bengaluru - KSPCB","[co, no2, o3, pm25, so2]",2016-03-22,2018-02-22
2,797,City Railway Station - KSPCB,"[co, no2, pm10, so2]",2016-03-21,2018-02-22
3,2589,SaneguravaHalli - KSPCB,"[co, no2, pm10, so2]",2016-03-22,2018-02-22
4,2592,"BWSSB Kadabesanahalli, Bengaluru - KSPCB","[co, no2, o3, pm25, so2]",2016-03-22,2018-02-22
5,5547,"BWSSB Kadabesanahalli, Bengaluru - CPCB","[co, no2, o3, pm10, pm25, so2]",2018-03-09,2022-08-02
6,5548,"BTM Layout, Bengaluru - CPCB","[co, co, no2, no2, o3, o3, pm10, pm10, pm25, p...",2018-03-09,2026-03-29
7,5574,"City Railway Station, Bengaluru - KSPCB","[co, co, no2, no2, pm10, pm10, so2, so2]",2018-03-09,2026-03-28
8,5607,"Peenya, Bengaluru - CPCB","[co, co, no2, no2, o3, o3, pm10, pm10, pm25, p...",2018-03-09,2026-03-29
9,5644,"Sanegurava Halli, Bengaluru - KSPCB","[co, co, no2, no2, pm10, pm10, so2, so2]",2018-03-09,2026-03-29


In [10]:
# See full table without truncation
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 20)

print(df_india[["id", "name", "useful_params", "from", "to"]].to_string(index=False))

     id                                     name                                                useful_params       from         to
    412                Peenya, Bengaluru - KSPCB                                         [co, no2, pm25, so2] 2016-03-22 2018-02-22
    594            BTM Layout, Bengaluru - KSPCB                                     [co, no2, o3, pm25, so2] 2016-03-22 2018-02-22
    797             City Railway Station - KSPCB                                         [co, no2, pm10, so2] 2016-03-21 2018-02-22
   2589                  SaneguravaHalli - KSPCB                                         [co, no2, pm10, so2] 2016-03-22 2018-02-22
   2592 BWSSB Kadabesanahalli, Bengaluru - KSPCB                                     [co, no2, o3, pm25, so2] 2016-03-22 2018-02-22
   5547  BWSSB Kadabesanahalli, Bengaluru - CPCB                               [co, no2, o3, pm10, pm25, so2] 2018-03-09 2022-08-02
   5548             BTM Layout, Bengaluru - CPCB [co, co, no2, no2, o3, o3, 

In [ ]:
import time
import json
from datetime import datetime

TARGET_PARAMS = {"pm25", "pm10", "no2", "so2", "o3", "co"}

STATIONS = {
    5548:  "BTM Layout",
    5574:  "City Railway Station",
    5607:  "Peenya",
    6973:  "Jayanagar",
    6974:  "Bapuji Nagar",
    6975:  "Silk Board",
    6983:  "Hombegowda Nagar",
    6984:  "Hebbal",
}

all_readings = []

for station_id, station_name in STATIONS.items():
    print(f"\nFetching: {station_name} (id={station_id})")
    page = 1
    station_total = 0

    while True:
        response = requests.get(
            f"https://api.openaq.org/v3/locations/{station_id}/sensors",
            headers={"X-API-Key": key},
        )

        # Get sensor IDs for this station
        sensors = response.json().get("results", [])
        sensor_ids = [
            s["id"] for s in sensors
            if s["parameter"]["name"] in TARGET_PARAMS
        ]
        break  # just need sensor IDs once

    for sensor_id in sensor_ids:
        page = 1
        while True:
            r = requests.get(
                f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/hourly",
                headers={"X-API-Key": key},
                params={
                    "date_from": "2018-01-01",
                    "date_to":   "2024-12-31",
                    "limit":     1000,
                    "page":      page,
                }
            )

            if r.status_code != 200:
                print(f"  Error {r.status_code} on sensor {sensor_id}, page {page}")
                break

            results = r.json().get("results", [])
            if not results:
                break

            for reading in results:
                all_readings.append({
                    "station_id":   station_id,
                    "station_name": station_name,
                    "sensor_id":    sensor_id,
                    "parameter":    reading["parameter"]["name"],
                    "value":        reading["value"],
                    "date":         reading["period"]["datetimeFrom"]["local"][:10],
                })

            station_total += len(results)
            print(f"  sensor {sensor_id} · page {page} · {station_total} rows so far", end="\r")
            page += 1
            time.sleep(0.5)

print(f"\n\nDone. Total raw readings: {len(all_readings):,}")
df_raw = pd.DataFrame(all_readings)
df_raw.head()


Fetching: BTM Layout (id=5548)
